In [ ]:

# %pip install langgraph langchain-core langchain-openai
# %pip install rich python-dotenv
# %pip install tavily-python langchain-tavily

### Defining state

 - Think of this as the "contract" for your entire pipeline.
 - Every node can read ANY field, and return updates to ANY field.

 - Why TypedDict and not a regular dict?
    - It gives you type hints, autocomplete, and catches typos.
    - LangGraph uses it to understand what your state looks like.

In [14]:
from typing import TypedDict, Optional, Any

class ContentState(TypedDict):
    # ── Core content fields ──────────────────────────────
    topic:           str            # The topic to research and write about
    research_notes:  str            # Filled by Researcher
    draft:           str            # Filled by Writer (updated each revision)
    review_score:    Optional[int]  # Filled by Reviewer
    feedback:        str            # Filled by Reviewer
    
    # ── New fields for revision tracking ─────────────────
    revision_count:   int           # How many times Writer has been called
    review_data:      Any           # Full JSON review breakdown (dict)
    revision_history: list          # Log of [{revision, score, feedback}]

In [ ]:
import json
import re
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from rich import print as rprint

load_dotenv()  

api_key = os.getenv("OPEN_ROUTER_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")


llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free", 
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:3000", # Required by OpenRouter for some models
        "X-Title": "LangGraph App",
    }
)

### Researcher

In [16]:
def researcher_node(state: ContentState) -> dict:
    """
    The Researcher node:
    - Reads:  'topic' from state
    - Writes: 'research_notes' to state
    
    How it works:
    1. LLM with tools decides WHAT to search
    2. We execute the search
    3. LLM synthesizes results into clean research notes
    
    This is the "ReAct" pattern:
    Reason → Act (use tool) → Observe results → Reason again
    """
    print("\n🔍 RESEARCHER NODE ACTIVATED")
    
    
    topic = state["topic"]
    
    # ── Step A: Set up LLM with Tool ─────────────────────────
    search_tool = TavilySearch(max_results=5)
    llm_with_tools = llm.bind_tools([search_tool])
    
    # ── Step B: Ask the LLM what to search for ───────────────
    print(f"📌 Topic to research: {topic}")
    
    messages = [
        SystemMessage(content="""You are a research assistant. 
Your job is to search for information about the given topic.
Use the search tool to find relevant, up-to-date information."""),
        
        HumanMessage(content=f"""Research this topic thoroughly: {topic}
        
Please search for:
1. Key concepts and definitions
2. Current trends and developments  
3. Important facts and statistics
4. Expert opinions or analysis

Use the search tool now."""),
    ]
    
    # First LLM call: LLM decides to call the tool
    print("🤖 LLM deciding what to search for...")
    llm_response = llm_with_tools.invoke(messages)
    
    print(f"🔧 Tool calls requested: {len(llm_response.tool_calls)}")
    
    # ── Step C: Execute the Tool Calls ───────────────────────
    # The LLM told us WHAT to search — now we actually DO the search
    
    all_search_results = []  # Collect all raw results here
    tool_messages = []       # These go back to the LLM for synthesis
    
    for tool_call in llm_response.tool_calls:
        print(f"🌐 Executing search: '{tool_call['args'].get('query', 'N/A')}'")
        
        # Actually run the Tavily search
        search_results = search_tool.invoke(tool_call["args"])
        
        # search_results is a list of dicts with keys:
        # 'url', 'content', 'title', 'score'
        all_search_results.append(search_results)
        
        print(f"   ✅ Got {len(search_results)} results")
        
        # Wrap results in a ToolMessage so the LLM can read them
        # The tool_call_id links this result back to the specific tool call
        tool_messages.append(
            ToolMessage(
                content=str(search_results),
                tool_call_id=tool_call["id"],
            )
        )
    
    # ── Step D: LLM Synthesizes the Results ──────────────────
    # Now feed the raw search results BACK to the LLM
    # Ask it to write clean, structured research notes
    
    print("📝 LLM synthesizing search results into research notes...")
    
    # Build the full conversation: original messages + LLM response + tool results
    synthesis_messages = messages + [llm_response] + tool_messages + [
        HumanMessage(content=f"""Based on these search results, write comprehensive 
research notes about: {topic}

Structure your notes as:
## Key Findings
[main points discovered]

## Important Facts & Data  
[specific statistics, dates, numbers]

## Current Trends
[what's happening now]

## Summary
[2-3 sentence overview]

Be specific and cite the sources where relevant.""")
    ]
    
    # Second LLM call: pure synthesis, no more tool calls needed
    final_response = llm.invoke(synthesis_messages)
    
    research_notes = final_response.content
    
    print(f"✅ Research complete! Notes: {len(research_notes)} characters")
    print("─" * 50)
    
    return {"research_notes": research_notes}

### The writer

In [17]:


def writer_node(state: ContentState) -> dict:
    """
    UPDATED Writer node — now fully revision-aware.
    
    - Reads:  'topic', 'research_notes', 'feedback', 
              'draft' (previous), 'revision_count', 'revision_history'
    - Writes: 'draft', 'revision_count', 'revision_history'
    
    Two modes:
      Mode A (First draft):  revision_count == 0, no feedback yet
      Mode B (Revision):     revision_count > 0, has specific feedback to address
    """
    print("\n✍️  WRITER NODE ACTIVATED")
    
    
    topic            = state["topic"]
    research_notes   = state["research_notes"]
    feedback         = state.get("feedback", "")
    previous_draft   = state.get("draft", "")
    revision_count   = state.get("revision_count", 0)
    revision_history = state.get("revision_history", [])
    review_data      = state.get("review_data", {})
    
    print(f"📝 Draft attempt #{revision_count + 1}")
    
    # ── Choose the right prompt based on mode ────────────────
    if revision_count == 0:
        # ── MODE A: FIRST DRAFT ───────────────────────────────
        print("✨ Mode: First draft from research notes")
        
        prompt = f"""You are a professional content writer at a top publication.

TOPIC: {topic}

RESEARCH NOTES (your ONLY source of facts — use them!):
{research_notes}

YOUR TASK:
Write an exceptional article based on the research notes above.

REQUIREMENTS:
- Hook the reader in the first sentence
- Use specific facts, data points, and examples from the research notes
- Structure: Strong intro → 2-3 substantive body paragraphs → Memorable conclusion
- Write for an intelligent general audience (no unnecessary jargon)
- Be specific, not generic — avoid clichés like "In today's fast-paced world..."
- Length: 400-600 words

Write the article now:"""

    else:
        # ── MODE B: REVISION ──────────────────────────────────
        print(f"🔄 Mode: Revision #{revision_count} — addressing editor feedback")
        
        # Build a history summary so the writer knows what's been tried
        history_summary = ""
        if revision_history:
            history_summary = "\n\nPREVIOUS REVISION SCORES:\n"
            for h in revision_history:
                history_summary += f"  Revision {h['revision']}: {h['score']}/10\n"
        
        # Extract specific category scores if available
        score_breakdown = ""
        if review_data and "scores" in review_data:
            scores = review_data["scores"]
            score_breakdown = f"""
SPECIFIC SCORE BREAKDOWN (focus on the low scores!):
  Accuracy:     {scores.get('accuracy', '?')}/2
  Clarity:      {scores.get('clarity', '?')}/2  
  Structure:    {scores.get('structure', '?')}/2
  Engagement:   {scores.get('engagement', '?')}/2
  Completeness: {scores.get('completeness', '?')}/2"""
        
        strengths_text = ""
        if review_data and "strengths" in review_data:
            strengths_text = "\nSTRENGTHS TO KEEP:\n" + "\n".join(
                f"  ✓ {s}" for s in review_data["strengths"]
            )
        
        issues_text = ""
        if review_data and "critical_issues" in review_data:
            issues_text = "\nCRITICAL ISSUES TO FIX:\n" + "\n".join(
                f"  ✗ {i}" for i in review_data["critical_issues"]
            )
        
        prompt = f"""You are a professional writer doing a REVISION.

TOPIC: {topic}

RESEARCH NOTES (your source of facts):
{research_notes}

{'─'*40}
YOUR PREVIOUS DRAFT (revision #{revision_count - 1}):
{previous_draft}

{'─'*40}
EDITOR'S FEEDBACK TO ADDRESS:
{feedback}
{score_breakdown}
{strengths_text}
{issues_text}
{history_summary}
{'─'*40}

YOUR TASK:
Rewrite the article to address ALL of the editor's feedback.

IMPORTANT RULES:
1. Keep what the editor praised (the strengths listed above)
2. Fix EVERY critical issue mentioned
3. Use more specific facts from the research notes
4. Do NOT just make small tweaks — if the editor wants big changes, make them
5. Aim for the same length (400-600 words) but make it significantly better
6. Do NOT mention this is a revision

Write the improved article now:"""

    print("🤖 LLM generating the draft...")
    response = llm.invoke(prompt)
    new_draft = response.content
    
    # ── Update revision history ───────────────────────────────
    # Log the PREVIOUS draft's results before overwriting
    if revision_count > 0 and previous_draft:
        revision_history = revision_history + [{
            "revision": revision_count,
            "score":    state.get("review_score", "N/A"),
            "feedback": feedback[:150] + "..." if len(feedback) > 150 else feedback,
        }]
    
    print(f"✅ Draft #{revision_count + 1} complete ({len(new_draft)} characters)")
    print("─" * 50)
    
    return {
        "draft":            new_draft,
        "revision_count":   revision_count + 1,
        "revision_history": revision_history,
    }

### Reviewer node

In [18]:
def reviewer_node(state: ContentState) -> dict:
    """
    UPGRADED Reviewer node — Strict Editor persona.
    
    - Reads:  'draft', 'topic', 'research_notes', 'revision_count'
    - Writes: 'review_score', 'feedback'
    
    Scoring rubric (each category 0-2 points = 10 points total):
      1. Accuracy      — Does it match the research notes?
      2. Clarity       — Is it easy to understand?
      3. Structure     — Does it flow logically?
      4. Engagement    — Would a reader keep reading?
      5. Completeness  — Does it cover the topic fully?
    """
    print("\n🔎 REVIEWER NODE ACTIVATED")
    
    # Track which revision this is
    revision_count = state.get("revision_count", 0)
    print(f"📋 Reviewing draft (revision #{revision_count})")
    
    draft          = state["draft"]
    topic          = state["topic"]
    research_notes = state.get("research_notes", "")
    
    # ── The Strict Editor System Prompt ───────────────────────────────
    system_prompt = """You are ALEX, a ruthlessly strict senior editor at a top publication.

    Your reputation depends on only publishing exceptional content.
    You do NOT give high scores easily — a 10 is a masterpiece, a 7 means real problems exist.

    Your scoring rubric (each category scored 0-2):
    ┌─────────────────┬────────────────────────────────────────────────┐
    │ Category        │ What you're checking                           │
    ├─────────────────┼────────────────────────────────────────────────┤
    │ Accuracy (0-2)  │ Facts align with research, no hallucinations   │
    │ Clarity (0-2)   │ Plain language, no jargon without explanation  │
    │ Structure (0-2) │ Clear intro, body, conclusion — logical flow   │
    │ Engagement (0-2)│ Hook, storytelling, would reader continue?     │
    │ Completeness(0-2)│ Covers topic fully, no obvious gaps           │
    └─────────────────┴────────────────────────────────────────────────┘
    Total score = sum of all categories (max 10)

    You MUST respond in valid JSON. No text before or after the JSON block."""

    # ── The Review Prompt ─────────────────────────────────────────────
    review_prompt = f"""Review this article draft.

    ASSIGNED TOPIC: {topic}

    RESEARCH NOTES THAT SHOULD BE USED:
    {research_notes}

    DRAFT TO REVIEW:
    {draft}

    Respond ONLY with this exact JSON structure:
    {{
        "scores": {{
            "accuracy": <0-2>,
            "clarity": <0-2>,
            "structure": <0-2>,
            "engagement": <0-2>,
            "completeness": <0-2>
        }},
        "total_score": <sum of all scores, 0-10>,
        "strengths": ["<specific strength 1>", "<specific strength 2>"],
        "critical_issues": ["<specific issue 1>", "<specific issue 2>"],
        "feedback": "<detailed, actionable feedback telling the writer EXACTLY what to fix>"
    }}"""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=review_prompt),
    ]
    
    print("🤖 Editor Alex reviewing the draft...")
    response = llm.invoke(messages)
    raw_response = response.content
    
    # ── Robust JSON Parsing ───────────────────────────────────────────
    # LLMs sometimes add markdown fences around JSON — handle that
    review_data = _parse_review_json(raw_response)
    
    score    = review_data["total_score"]
    feedback = review_data["feedback"]
    
    # ── Display the Review Report ─────────────────────────────────────
    _display_review_report(review_data, revision_count)
    
    return {
        "review_score": score,
        "feedback":     feedback,
        "review_data":  review_data,   # Store full breakdown in state
    }


def _parse_review_json(raw_response: str) -> dict:
    """
    Robustly parse the JSON response from the reviewer LLM.
    Handles markdown code fences, extra whitespace, etc.
    """
    # Strategy 1: Direct JSON parse
    try:
        return json.loads(raw_response.strip())
    except json.JSONDecodeError:
        pass
    
    # Strategy 2: Extract JSON from markdown code fences
    # Matches ```json ... ``` or ``` ... ```
    json_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw_response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Strategy 3: Find anything that looks like a JSON object
    json_match = re.search(r"\{.*\}", raw_response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    
    # Strategy 4: Full fallback — parse score from text manually
    print("⚠️  JSON parsing failed — using fallback parser")
    score = 5  # Default middle score
    
    # Try to find a number after "score" or "total"
    score_match = re.search(r'(?:total_score|score)["\s:]+(\d+)', raw_response, re.IGNORECASE)
    if score_match:
        score = int(score_match.group(1))
    
    return {
        "scores": {"accuracy": 1, "clarity": 1, "structure": 1, "engagement": 1, "completeness": 1},
        "total_score": score,
        "strengths": ["Could not parse strengths"],
        "critical_issues": ["Could not parse issues"],
        "feedback": raw_response  # Use the full raw response as feedback
    }


def _display_review_report(review_data: dict, revision_count: int):
    """Pretty-print the review report to the console."""
    
    score   = review_data["total_score"]
    scores  = review_data.get("scores", {})
    
    # Score bar visualization
    filled = "█" * score
    empty  = "░" * (10 - score)
    bar    = f"{filled}{empty}"
    
    # Verdict based on score
    if score >= 8:
        verdict = "✅ APPROVED"
    elif score >= 6:
        verdict = "⚠️  NEEDS REVISION"
    else:
        verdict = "❌ SIGNIFICANT REVISION REQUIRED"
    
    print(f"\n{'─'*50}")
    print(f"  📊 EDITORIAL REVIEW REPORT (Revision #{revision_count})")
    print(f"{'─'*50}")
    print(f"  Overall Score: [{bar}] {score}/10")
    print(f"  Verdict:       {verdict}")
    print(f"{'─'*50}")
    
    if scores:
        print("  Category Breakdown:")
        categories = {
            "accuracy":     "Accuracy    ",
            "clarity":      "Clarity     ",
            "structure":    "Structure   ",
            "engagement":   "Engagement  ",
            "completeness": "Completeness",
        }
        for key, label in categories.items():
            cat_score = scores.get(key, "?")
            cat_bar   = "●" * int(cat_score) + "○" * (2 - int(cat_score))
            print(f"    {label}: [{cat_bar}] {cat_score}/2")
    
    print(f"\n  ✨ Strengths:")
    for s in review_data.get("strengths", []):
        print(f"    + {s}")
    
    print(f"\n  🚨 Critical Issues:")
    for issue in review_data.get("critical_issues", []):
        print(f"    - {issue}")
    
    print(f"\n  📝 Editor's Feedback:")
    print(f"    {review_data.get('feedback', 'No feedback')[:200]}...")
    print(f"{'─'*50}")

### Conditional edge logic

In [19]:
# ============================================================
# THE ROUTING FUNCTION — The Brain of the Loop
# ============================================================
# This is NOT a node. It's a pure routing function.
# LangGraph calls it after the Reviewer runs and uses the
# RETURN VALUE to decide which node executes next.
# ============================================================

# Configuration — easy to change
APPROVAL_THRESHOLD  = 8    # Score needed to pass review
MAX_REVISION_LIMIT  = 3    # Safety net: stop looping after this many revisions

def route_after_review(state: ContentState) -> str:
    """
    Conditional routing function — called by LangGraph after Reviewer.
    
    Returns a STRING that maps to a node name (or END).
    
    Possible return values:
      "revise"    → sends flow back to writer node
      "approved"  → sends flow to END
    
    ⚠️  The string values MUST match the keys in the 
        add_conditional_edges() mapping dict below.
    """
    score          = state.get("review_score", 0)
    revision_count = state.get("revision_count", 0)
    
    print(f"\n🔀 ROUTING DECISION")
    print(f"   Score:          {score}/10")
    print(f"   Threshold:      {APPROVAL_THRESHOLD}/10")
    print(f"   Revision count: {revision_count}/{MAX_REVISION_LIMIT}")
    
    # ── Guard: Maximum revisions safety net ──────────────────
    # Prevents infinite loops if the LLM keeps getting low scores
    if revision_count >= MAX_REVISION_LIMIT:
        print(f"   Decision: ⏹️  MAX REVISIONS REACHED — forcing END")
        print(f"   (Final score: {score}/10 — did not meet threshold)")
        return "approved"   # Exit even if score is low
    
    # ── Main routing logic ────────────────────────────────────
    if score >= APPROVAL_THRESHOLD:
        print(f"   Decision: ✅ APPROVED — score {score} >= {APPROVAL_THRESHOLD}")
        return "approved"
    else:
        gap = APPROVAL_THRESHOLD - score
        print(f"   Decision: 🔄 REVISE — score {score} is {gap} points below threshold")
        return "revise"

### Building blocks

In [20]:
# ============================================================
# BUILD THE COMPLETE GRAPH WITH THE REVISION LOOP
# ============================================================

from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(ContentState)

# ── Register all 3 nodes ─────────────────────────────────
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("writer",     writer_node)
graph_builder.add_node("reviewer",   reviewer_node)

# ── Linear edges (always happen) ─────────────────────────
graph_builder.add_edge(START,        "researcher")
graph_builder.add_edge("researcher", "writer")
graph_builder.add_edge("writer",     "reviewer")

# ── THE CONDITIONAL EDGE (the magic loop) ────────────────
#
#  reviewer ──────────────────────────────────────► END
#      │                                          (approved)
#      └──────────────────────────────────────► writer
#                                              (revise)
#
graph_builder.add_conditional_edges(
    "reviewer",           # After this node runs...
    route_after_review,   # ...call this function...
    {                     # ...and route based on the return value
        "revise":   "writer",  # Back to writer for another attempt
        "approved": END,       # Done! Exit the graph
    }
)

# ── Compile ───────────────────────────────────────────────
graph = graph_builder.compile()

print("✅ Graph with revision loop compiled successfully!")

✅ Graph with revision loop compiled successfully!


### Visualize the graph

In [ ]:
# LangGraph can render your graph as an image — super useful for debugging!

from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # If the image rendering fails, print the text version
    print("Graph structure (text):")
    graph.get_graph().print_ascii()
    print(f"\n(Image rendering requires additional deps: {e})")

### Run

In [22]:
# ============================================================
# FULL EXECUTION — Watch the self-improvement loop in action
# ============================================================

initial_state = {
    "topic":            "How quantum computing will break modern encryption and what we're doing about it",
    "research_notes":   "",
    "draft":            "",
    "review_score":     None,
    "feedback":         "",
    "revision_count":   0,
    "review_data":      {},
    "revision_history": [],
}

print("🚀 AUTONOMOUS CONTENT ENGINE — FULL PIPELINE WITH REVISION LOOP")
print("=" * 65)
print(f"📌 Topic:             {initial_state['topic']}")
print(f"🎯 Approval threshold: {APPROVAL_THRESHOLD}/10")
print(f"🛡️  Max revisions:     {MAX_REVISION_LIMIT}")
print("=" * 65)

# Run the graph — it will loop automatically until approved
final_state = graph.invoke(initial_state)

🚀 AUTONOMOUS CONTENT ENGINE — FULL PIPELINE WITH REVISION LOOP
📌 Topic:             How quantum computing will break modern encryption and what we're doing about it
🎯 Approval threshold: 8/10
🛡️  Max revisions:     3

🔍 RESEARCHER NODE ACTIVATED
📌 Topic to research: How quantum computing will break modern encryption and what we're doing about it
🤖 LLM deciding what to search for...
🔧 Tool calls requested: 1
🌐 Executing search: 'quantum computing breaking modern encryption'
   ✅ Got 7 results
📝 LLM synthesizing search results into research notes...
✅ Research complete! Notes: 5047 characters
──────────────────────────────────────────────────

✍️  WRITER NODE ACTIVATED
📝 Draft attempt #1
✨ Mode: First draft from research notes
🤖 LLM generating the draft...
✅ Draft #1 complete (3380 characters)
──────────────────────────────────────────────────

🔎 REVIEWER NODE ACTIVATED
📋 Reviewing draft (revision #1)
🤖 Editor Alex reviewing the draft...

─────────────────────────────────────────────────

In [24]:
# ============================================================
# COMPREHENSIVE RESULTS REPORT
# ============================================================

def display_final_report(state: dict):
    """Display a complete report of the pipeline results."""
    
    width = 65
    
    print("\n" + "█" * width)
    print("  📰 AUTONOMOUS CONTENT ENGINE — FINAL REPORT")
    print("█" * width)
    
    # ── Journey Summary ──────────────────────────────────────
    revision_history = state.get("revision_history", [])
    final_score      = state.get("review_score", 0)
    total_revisions  = state.get("revision_count", 1) - 1
    
    print(f"\n  📌 Topic:           {state['topic']}")
    print(f"  📝 Total drafts:    {state['revision_count']}")
    print(f"  🔄 Revisions made:  {total_revisions}")
    print(f"  ⭐ Final score:     {final_score}/10")
    
    approved = final_score >= APPROVAL_THRESHOLD
    print(f"  📋 Status:         {'✅ APPROVED' if approved else '⏹️  MAX REVISIONS REACHED'}")
    
    # ── Score progression ─────────────────────────────────────
    if revision_history:
        print(f"\n  📈 SCORE PROGRESSION:")
        for h in revision_history:
            score      = h['score']
            score_bar  = "█" * score + "░" * (10 - score)
            print(f"     Draft {h['revision']}: [{score_bar}] {score}/10")
        
        # Final score
        final_bar = "█" * final_score + "░" * (10 - final_score)
        print(f"     Draft {state['revision_count']}: [{final_bar}] {final_score}/10 ← FINAL")
    
    # ── Final Review Breakdown ────────────────────────────────
    review_data = state.get("review_data", {})
    if review_data and "scores" in review_data:
        print(f"\n  📊 FINAL REVIEW BREAKDOWN:")
        for category, score in review_data["scores"].items():
            bar = "●" * int(score) + "○" * (2 - int(score))
            print(f"     {category.capitalize():15}: [{bar}] {score}/2")
    
    # ── Research Notes ────────────────────────────────────────
    print(f"\n{'─'*width}")
    print(f"  🔍 RESEARCH NOTES (summarized)")
    print(f"{'─'*width}")
    notes = state.get("research_notes", "")
    # Show first 500 chars of research notes
    print(f"  {notes[:500]}{'...' if len(notes) > 500 else ''}")
    
    # ── Final Draft ───────────────────────────────────────────
    print(f"\n{'─'*width}")
    print(f"  ✍️  FINAL APPROVED DRAFT")
    print(f"{'─'*width}")
    rprint(state.get("draft", "No draft generated"))
    
    # ── Editor's Final Feedback ───────────────────────────────
    print(f"\n{'─'*width}")
    print(f"  💬 EDITOR'S FINAL FEEDBACK")
    print(f"{'─'*width}")
    print(f"  {state.get('feedback', 'No feedback')}")
    
    print(f"\n{'█'*width}\n")


display_final_report(final_state)


█████████████████████████████████████████████████████████████████
  📰 AUTONOMOUS CONTENT ENGINE — FINAL REPORT
█████████████████████████████████████████████████████████████████

  📌 Topic:           How quantum computing will break modern encryption and what we're doing about it
  📝 Total drafts:    1
  🔄 Revisions made:  0
  ⭐ Final score:     9/10
  📋 Status:         ✅ APPROVED

  📊 FINAL REVIEW BREAKDOWN:
     Accuracy       : [●●] 2/2
     Clarity        : [●●] 2/2
     Structure      : [●●] 2/2
     Engagement     : [●●] 2/2
     Completeness   : [●○] 1/2

─────────────────────────────────────────────────────────────────
  🔍 RESEARCH NOTES (summarized)
─────────────────────────────────────────────────────────────────
  ## Key Findings  
- **Shor’s algorithm** enables a sufficiently powerful quantum computer to factor large integers exponentially faster than classical machines, rendering RSA, Elliptic‑Curve Cryptography (ECC), and most public‑key schemes obsolete【1†L3-L4】【2†L1-L3】

Imagine waking up to find that every secret you ever sent online could be read by anyone with a quantum computer. 
The threat is not speculative; it follows directly from two quantum algorithms that could dismantle the 
cryptographic foundations of the internet. Shor’s algorithm, when run on a sufficiently powerful machine, factors 
large integers exponentially faster than any classical approach, which would instantly break RSA, elliptic‑curve 
cryptography and most public‑key schemes in use today【1†L3-L4】【2†L1-L3】. Grover’s algorithm, meanwhile, gives a
quadratic speed‑up against symmetric ciphers, effectively halving their security strength—AES‑256 would offer 
roughly the protection of a 128‑bit key against a quantum attacker, while AES‑128 would drop to about 64‑bit 
security【1†L13-L14】. Because adversaries already practice “harvest now, decrypt later,” they are intercepting and
storing today’s encrypted traffic with the expectation of cracking it once a cryptographically relevant quantum 
computer (CRQC) appears【1†L5-L7】【4†L1-L3】.

Estimates for when a CRQC capable of breaking current public‑key crypto might arrive converge on the 2030s, with 
the RAND Corporation noting that such a machine is “not before the 2030s”【3†L1-L4】. Some industry voices, 
however, warn the timeline could be much shorter: Google has suggested that quantum computers could break most 
existing encryption by the end of this decade, around 2029【5†L1-L4】. Even if the hardware needed for Shor’s 
algorithm—millions of error‑corrected qubits—remains far beyond today’s noisy processors, which sit below 1,000 
qubits【3†L2-L3】, the mere possibility of a future breakthrough is enough to motivate action now. The threat is 
therefore both a long‑term strategic concern and an immediate incentive to protect data that must stay secret for 
decades.

In response, governments and standards bodies are accelerating the transition to post‑quantum cryptography (PQC). 
The U.S. National Institute of Standards and Technology is finalizing a suite of PQC algorithms drawn from 
lattice‑, hash‑, code‑ and multivariate‑based constructions, with draft standards already out for public comment 
and final specifications expected in 2024‑2025【3†L8-L11】. Complementing this, National Security Memorandum 10 
directs federal agencies to begin migrating to PQC and to mitigate as much quantum risk as possible【3†L8-L11】. 
Industry leaders are adopting “crypto‑agility”—designing systems that can swap cryptographic primitives without 
major overhauls—and testing hybrid schemes that combine classical and post‑quantum keys during the transition. Many
organizations are also boosting symmetric‑key sizes, relying on AES‑256 as a quantum‑resistant stopgap while they 
await PQC deployment【1†L13-L14】. Investment in quantum‑resistant security has surged, with venture capital 
flowing into startups that offer PQC libraries, hardware security modules and even quantum‑key distribution pilots.

The quantum era is not a distant sci‑fi scenario; it is a calculable risk that already shapes how we safeguard 
information. By embracing crypto‑agility, upgrading to larger symmetric keys, and preparing for the imminent 
arrival of standardized post‑quantum algorithms, we can turn a looming vulnerability into a managed transition. The
window to act is narrowing, but the tools to protect our digital future are already in hand.


─────────────────────────────────────────────────────────────────
  💬 EDITOR'S FINAL FEEDBACK
─────────────────────────────────────────────────────────────────
  Add a short section after the NSM-10 discussion that notes the EU’s Cybersecurity Act updates and the UK’s NCSC roadmap (target 2035) to show global momentum. Follow this with a concrete example of how financial firms, cloud providers, and companies like Google, Microsoft, and JPMorgan are conducting risk assessments and testing hybrid schemes that combine classical keys with PQC‑protected key exchange. Finally, include a sentence on current advances in quantum error correction and alternative architectures (trapped ions, photonic qubits) that are shortening the path to a cryptographically relevant quantum computer. These additions will fill the noticeable gaps and make the article’s coverage of current trends more complete.

█████████████████████████████████████████████████████████████████

